# Simulating Population Shift to Expose Model Drift in Life Insurance Underwriting

This notebook demonstrates how to use [DataFramer](https://dataframer.ai) to detect and analyze model drift in a life insurance risk model trained on 2015 data.

## What we'll do

1. Load the Prudential training data and fit a simple risk classifier (our "frozen 2015 model")
2. Use DataFramer to build a spec from the 2015 population
3. Shift the spec to simulate a 2026 applicant pool
4. Run both populations through the frozen 2015 model and compare risk score distributions
5. Frame the business question the model cannot answer on its own

> **Prerequisite:** `train.csv` is sourced from [AntonUBC's GitHub repository](https://github.com/AntonUBC) and is kept in the `files/` folder of this project.

## Setup

In [ ]:
import os

!git clone https://github.com/aimonlabs/dataframer-docs-public.git
os.chdir("dataframer-docs-public/insurance_underwriting_model_drift_detection")

In [ ]:
!pip install xgboost scikit-learn matplotlib pandas pydataframer tenacity pyyaml requests scipy

import copy
import io
import os
import zipfile
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import requests
import yaml
from scipy.optimize import minimize
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import train_test_split
from tenacity import retry, retry_if_result, stop_never, wait_fixed
from xgboost import XGBRegressor

import dataframer
from dataframer import Dataframer

df_client = Dataframer(api_key=os.environ["DATAFRAMER_API_KEY"])
FILES_DIR = Path("files")

print(f"DataFramer SDK version: {dataframer.__version__}")

## 1. Load the Data

The Prudential dataset has 128 columns — mostly anonymized medical and employment variables. We select 29 columns and add one derived feature, staying under DataFramer's 40-column seed limit while retaining enough signal for the model.

| Column(s) | What it represents |
|---|---|
| `Ins_Age`, `Ht`, `Wt`, `BMI` | Applicant biometrics, each normalized to [0, 1] |
| `Product_Info_1–4` | Insurance product attributes (type, sub-type, channel, coverage amount) — anonymized |
| `Employment_Info_1–3, 6` | Employment status, duration, and income-related attributes — anonymized |
| `InsuredInfo_1–4, 6–7` | Characteristics of the insured person (e.g. smoker status, education, occupation) — anonymized |
| `Insurance_History_1–2, 4–5` | Prior insurance coverage and claim history — anonymized |
| `Family_Hist_1–5` | Family medical and demographic history — anonymized |
| `Medical_History_1` | The single highest-importance medical history variable (selected by feature importance) — anonymized |
| `Med_Keywords_Count` | Count of the 48 binary `Medical_Keyword_*` flags that are set for an applicant — a single integer summarizing medical condition breadth |
| `Response` | **Target.** Underwriter's risk class, ordinal 1–8 (1 = lowest risk, 8 = highest risk) |

In [ ]:
raw_df = pd.read_csv(FILES_DIR / "train.csv")

# Aggregate the 48 binary Medical_Keyword columns into a single count
keyword_cols = [c for c in raw_df.columns if c.startswith("Medical_Keyword_")]
raw_df["Med_Keywords_Count"] = raw_df[keyword_cols].sum(axis=1)

FEATURE_COLS = [
    # Biometrics
    "Ins_Age", "Ht", "Wt", "BMI",
    # Product
    "Product_Info_1", "Product_Info_2", "Product_Info_3", "Product_Info_4",
    # Employment
    "Employment_Info_1", "Employment_Info_2", "Employment_Info_3", "Employment_Info_6",
    # Insured profile
    "InsuredInfo_1", "InsuredInfo_2", "InsuredInfo_3", "InsuredInfo_4", "InsuredInfo_6", "InsuredInfo_7",
    # Insurance history
    "Insurance_History_1", "Insurance_History_2", "Insurance_History_4", "Insurance_History_5",
    # Family history
    "Family_Hist_1", "Family_Hist_2", "Family_Hist_3", "Family_Hist_4", "Family_Hist_5",
    # Medical (aggregated)
    "Med_Keywords_Count",
    # Top medical history feature
    "Medical_History_1",
]
TARGET_COL = "Response"

df = raw_df[FEATURE_COLS + [TARGET_COL]].copy()
print(f"Dataset: {len(df):,} rows x {len(df.columns)} columns")
print(f"\nResponse class distribution:")
print(df[TARGET_COL].value_counts().sort_index().to_string())
df.head(3)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

counts = df[TARGET_COL].value_counts().sort_index()
axes[0].bar(counts.index, counts.values, color="#4C72B0", edgecolor="white")
axes[0].set_title("Risk Class Distribution")
axes[0].set_xlabel("Response Class (1=low, 8=high risk)")
axes[0].set_ylabel("Count")

axes[1].hist(df["Ins_Age"].dropna(), bins=40, color="#4C72B0", edgecolor="white")
axes[1].set_title("Age Distribution (normalized)")
axes[1].set_xlabel("Ins_Age")

axes[2].hist(df["BMI"].dropna(), bins=40, color="#4C72B0", edgecolor="white")
axes[2].set_title("BMI Distribution (normalized)")
axes[2].set_xlabel("BMI")

kw_counts = df["Med_Keywords_Count"].value_counts().sort_index()
axes[3].bar(kw_counts.index[:16], kw_counts.values[:16], color="#4C72B0", edgecolor="white")
axes[3].set_title("Medical Keyword Count")
axes[3].set_xlabel("Med_Keywords_Count")

plt.suptitle("2015 Applicant Population — Baseline Profile", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 2. Train the 2015 Underwriting Model

We train an XGBoost regressor on the selected features, then learn 7 class thresholds that map the continuous score to Prudential's 8-level risk scale. The thresholds are calibrated on the training set — this is the "2015 calibration" that we'll later apply unchanged to the 2026 population.

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Split Product_Info_2 (e.g. "D2" -> char_code=3, num=2)
    def _char(x):
        return ord(x[0].upper()) - ord("A") if isinstance(x, str) and len(x) >= 1 else -1

    def _num(x):
        try:
            return int(x[1:]) if isinstance(x, str) and len(x) >= 2 else -1
        except ValueError:
            return -1

    df["Product_Info_2_char"] = df["Product_Info_2"].apply(_char)
    df["Product_Info_2_num"] = df["Product_Info_2"].apply(_num)
    df = df.drop(columns=["Product_Info_2"])

    # BMI x Age interaction
    df["BMI_Age"] = df["BMI"] * df["Ins_Age"]

    # Label-encode any remaining string columns
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = pd.Categorical(df[col]).codes

    df = df.fillna(-1)
    return df

In [ ]:
X = engineer_features(df[FEATURE_COLS])
y = df[TARGET_COL].values

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)

train_preds = model.predict(X_train)
val_preds = model.predict(X_val)


def learn_thresholds(preds: np.ndarray, labels: np.ndarray, n_classes: int = 8) -> np.ndarray:
    """Learn 7 thresholds that map a continuous score to classes 1-8 by maximising QWK."""
    initial = np.percentile(preds, np.linspace(100 / n_classes, 100 - 100 / n_classes, n_classes - 1))

    def loss(thresholds):
        thresholds = np.sort(thresholds)
        pred_classes = np.clip(np.digitize(preds, thresholds) + 1, 1, n_classes)
        return -cohen_kappa_score(labels, pred_classes, weights="quadratic")

    result = minimize(loss, initial, method="Nelder-Mead", options={"maxiter": 5000, "xatol": 1e-5})
    return np.sort(result.x)


def predict_classes(preds: np.ndarray, thresholds: np.ndarray) -> np.ndarray:
    return np.clip(np.digitize(preds, thresholds) + 1, 1, 8)


THRESHOLDS = learn_thresholds(train_preds, y_train)

val_classes = predict_classes(val_preds, THRESHOLDS)
kappa = cohen_kappa_score(y_val, val_classes, weights="quadratic")
print(f"Validation quadratic weighted kappa: {kappa:.4f}")
print(f"Learned class thresholds: {THRESHOLDS.round(3)}")

In [ ]:
feat_names = list(X_train.columns)
importances = model.feature_importances_
top_idx = np.argsort(importances)[-20:]

fig, ax = plt.subplots(figsize=(8, 7))
ax.barh([feat_names[i] for i in top_idx], importances[top_idx], color="#4C72B0")
ax.set_title("Top 20 Feature Importances — 2015 Model")
ax.set_xlabel("Importance Score")
plt.tight_layout()
plt.show()

## 3. Profile the 2015 Applicant Population

We score the entire training population with the 2015 model and record the class distribution. These distributions become our baseline — the "expected" shape of the applicant book as the model was designed to handle it.

In [ ]:
all_preds_2015 = model.predict(engineer_features(df[FEATURE_COLS]))
classes_2015 = predict_classes(all_preds_2015, THRESHOLDS)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

counts_2015 = pd.Series(classes_2015).value_counts().sort_index()
axes[0].bar(counts_2015.index, counts_2015.values / len(classes_2015), color="#4C72B0", edgecolor="white")
axes[0].set_title("2015: Predicted Risk Classes")
axes[0].set_xlabel("Risk Class")
axes[0].set_ylabel("Proportion")
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

axes[1].hist(df["Ins_Age"].dropna(), bins=30, color="#4C72B0", edgecolor="white", density=True)
axes[1].set_title("2015: Age")
axes[1].set_xlabel("Ins_Age (normalized)")
axes[1].set_ylabel("Density")

axes[2].hist(df["BMI"].dropna(), bins=30, color="#4C72B0", edgecolor="white", density=True)
axes[2].set_title("2015: BMI")
axes[2].set_xlabel("BMI (normalized)")
axes[2].set_ylabel("Density")

kw_2015 = df["Med_Keywords_Count"].value_counts().sort_index()
axes[3].bar(kw_2015.index[:14], kw_2015.values[:14] / len(df), color="#4C72B0", edgecolor="white")
axes[3].set_title("2015: Medical Keyword Count")
axes[3].set_xlabel("Med_Keywords_Count")
axes[3].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

plt.suptitle("2015 Model — Baseline Population Profile", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

pct_high = (classes_2015 >= 7).mean() * 100
print(f"2015 baseline: {pct_high:.1f}% of applicants classified as high risk (class 7 or 8)")
print(f"Mean model score: {all_preds_2015.mean():.3f}  |  Median: {np.median(all_preds_2015):.3f}")

## 4. Build a DataFramer Seed and Specification

We upload the full training dataset as a seed. DataFramer analyses it and produces a Specification capturing the column structure, value distributions, and inter-feature dependencies.

We then edit that spec to describe a 2026 population shift: older applicants, higher BMI, more medical keyword flags.

In [14]:
seed_df = df.copy()
print(f"Seed dataset: {len(seed_df)} rows x {len(seed_df.columns)} columns")
print(f"\nResponse class distribution in seed:")
print(seed_df[TARGET_COL].value_counts().sort_index().to_string())

Seed dataset: 59381 rows x 30 columns

Response class distribution in seed:
Response
1     6207
2     6552
3     1013
4     1428
5     5432
6    11233
7     8027
8    19489


### Step 1: Upload Seed and Create Specification

In [ ]:
seed_csv = io.BytesIO(seed_df.to_csv(index=False).encode("utf-8"))
seed_csv.name = "insurance_seed.csv"

dataset = df_client.dataframer.seed_datasets.create_with_files(
    name=f"insurance_seed_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    description="Life insurance applicant seed dataset (2015 population)",
    dataset_type="SINGLE_FILE",
    files=[seed_csv],
)
dataset_id = dataset.id
print(f"Dataset ID: {dataset_id}")

spec = df_client.dataframer.specs.create(
    dataset_id=dataset_id,
    name=f"insurance_2015_spec_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    spec_generation_model_name="anthropic/claude-sonnet-4-6",
    generation_objectives=(""),
    extrapolate_values=True,
    generate_distributions=True,
)
spec_id = spec.id
print(f"Spec ID: {spec_id}")


def spec_not_ready(result):
    return result.status not in ("SUCCEEDED", "FAILED")


@retry(wait=wait_fixed(5), retry=retry_if_result(spec_not_ready), stop=stop_never)
def poll_spec(spec_id):
    return df_client.dataframer.specs.retrieve(spec_id=spec_id)


spec_result = poll_spec(spec_id)
if spec_result.status == "FAILED":
    raise RuntimeError("Spec analysis failed")
print("\nSpec ready!")

### Step 2: Review the Analysed Data Properties

In [19]:
import textwrap

spec = df_client.dataframer.specs.retrieve(spec_id=spec_id)
config = yaml.safe_load(spec.content_yaml)
spec_data = config.get("spec", config)

print("=" * 60)
print("SHARED REQUIREMENTS")
print("=" * 60)
req = spec_data.get("requirements") or spec_data.get("shared_requirements") or "(none)"
print("\n".join(textwrap.fill(line, 80) for line in req.splitlines()))

print()
print("=" * 60)
print("DATA PROPERTY VARIATIONS")
print("=" * 60)
for prop in spec_data.get("data_property_variations", []):
    values = prop.get("property_values", [])
    dists = prop.get("base_distributions", {})
    print(f"\n  {prop['property_name']}  ({len(values)} values)")
    for v in values:
        print(f"    {str(v)!r:45s}  weight: {dists.get(v, '-')}")

SHARED REQUIREMENTS
The data point represents an insurance applicant record with the following
fields: Ins_Age (normalized age between 0 and 1), Ht (normalized height between
0 and 1), Wt (normalized weight between 0 and 1), BMI (normalized BMI between 0
and 1), Product_Info_1 (integer product category), Product_Info_2 (alphanumeric
product code), Product_Info_3 (integer product subcategory), Product_Info_4
(normalized value between 0 and 1), Employment_Info_1 (normalized employment
duration), Employment_Info_2 (integer employment category), Employment_Info_3
(integer value), Employment_Info_6 (normalized value or missing), InsuredInfo_1
through InsuredInfo_7 (integer categorical values), Insurance_History_1,
Insurance_History_2, Insurance_History_4 (integer categorical values),
Insurance_History_5 (small decimal or missing), Family_Hist_1 (integer),
Family_Hist_2 through Family_Hist_5 (normalized decimal or missing),
Med_Keywords_Count (non-negative integer), Medical_History_1 (non-ne

### Step 3: Shift the Spec Toward a 2026 Population

The 2015 spec captures the seed's observed distributions. We now edit it to simulate how the applicant pool might look in 2026, using three evidence-based shifts:

- **Age (`Ins_Age`)**: +7 years across the population mean
- **BMI**: +9% of the current mean (reflecting rising obesity trends)
- **Medical complexity (`Med_Keywords_Count`)**: +50% of the current mean (more comorbidities per applicant)

Each property's distribution is shifted using **exponential tilting** — the minimum KL-divergence method that finds new weights `w'_i ∝ w_i · exp(λ · cᵢ)` such that the new weighted mean hits the exact target. This preserves the shape of the original distribution as much as possible rather than applying an arbitrary formula.

In [ ]:
import math

updated_spec_data = copy.deepcopy(spec_data)

# Ins_Age is stored as a normalized value: raw_age / 67, where raw age is an integer
# in [0, 67] as encoded in the Prudential dataset. A 7-year shift in raw age therefore
# corresponds to 7/67 ≈ 0.104 in normalized units. Update the denominator here if
# your version of the dataset uses a different normalization range.
INS_AGE_NORM_DENOMINATOR = 67

SHIFT_TARGETS = {
    "ins_age":            {"delta": 7 / INS_AGE_NORM_DENOMINATOR, "relative": False},  # +7 years absolute
    "bmi":                {"delta": 0.09,  "relative": True},   # +9% of current mean
    "med_keywords_count": {"delta": 0.50,  "relative": True},   # +50% of current mean
}

def parse_center(val):
    """Parse a numeric center from a bucket label: plain number or 'lo-hi' range."""
    s = str(val).strip()
    if "-" in s[1:]:
        lo, hi = s.rsplit("-", 1)
        return (float(lo) + float(hi)) / 2
    return float(s)

def bisect_lambda(f, lo=-50, hi=50, tol=1e-9, max_iter=200):
    """Binary search for λ such that f(λ) = 0."""
    for _ in range(max_iter):
        mid = (lo + hi) / 2
        if f(mid) < 0:
            lo = mid
        else:
            hi = mid
        if hi - lo < tol:
            break
    return (lo + hi) / 2

def shift_distribution(values, current_dist, target_mean):
    """
    Shift a discrete distribution to achieve target_mean using exponential tilting.

    Finds new weights w'_i = w_i * exp(λ * c_i) / Z where λ is the unique scalar
    that makes the new weighted mean equal to target_mean. This is the minimum
    KL-divergence solution: it moves the distribution as little as possible while
    hitting the exact target.
    """
    centers = [parse_center(v) for v in values]
    weights = [current_dist[v] / 100.0 for v in values]

    def tilted_mean(lam):
        tilted = [w * math.exp(lam * c) for w, c in zip(weights, centers)]
        Z = sum(tilted)
        return sum(c * t / Z for c, t in zip(centers, tilted)) - target_mean

    lam = bisect_lambda(tilted_mean)
    tilted = [w * math.exp(lam * c) for w, c in zip(weights, centers)]
    Z = sum(tilted)
    new_dist = {v: round(t / Z * 100, 1) for v, t in zip(values, tilted)}
    diff = round(100 - sum(new_dist.values()), 1)
    new_dist[values[-1]] = round(new_dist[values[-1]] + diff, 1)
    return new_dist

properties_shifted = []
for prop in updated_spec_data.get("data_property_variations", []):
    prop_name = prop["property_name"]
    prop_lower = prop_name.lower()
    target_key = next((k for k in SHIFT_TARGETS if k in prop_lower), None)
    if target_key is None:
        continue

    values = prop.get("property_values", [])
    current_dist = prop.get("base_distributions", {})
    if len(values) < 2 or not current_dist:
        continue

    centers = [parse_center(v) for v in values]
    weights = [current_dist.get(v, 0) / 100.0 for v in values]
    current_mean = sum(c * w for c, w in zip(centers, weights))

    cfg = SHIFT_TARGETS[target_key]
    target_mean = current_mean * (1 + cfg["delta"]) if cfg["relative"] else current_mean + cfg["delta"]

    new_dist = shift_distribution(values, current_dist, target_mean)
    prop["base_distributions"] = new_dist
    properties_shifted.append(prop_name)

    new_centers = [parse_center(v) for v in values]
    new_weights = [new_dist[v] / 100.0 for v in values]
    new_mean = sum(c * w for c, w in zip(new_centers, new_weights))
    print(f"Shifted '{prop_name}': mean {current_mean:.4f} → {new_mean:.4f} (target {target_mean:.4f})")
    for v, w in new_dist.items():
        print(f"  {str(v)!r:40s}  weight: {w:.1f}")

updated_yaml = yaml.dump({"spec": updated_spec_data}, allow_unicode=True, sort_keys=False)
updated_spec = df_client.dataframer.specs.update(spec_id=spec_id, content_yaml=updated_yaml)
print(f"\nSpec updated — ID: {updated_spec.id}")

## 5. Generate the "2026" Applicant Population

We generate 500 synthetic applicants from the updated spec. Each row has the same 30-column schema as the seed, but with distributions shifted toward the 2026 profile we described.

In [ ]:
run = df_client.dataframer.runs.create(
    spec_id=updated_spec.id,
    number_of_samples=500,
    generation_model="anthropic/claude-haiku-4-5",
    outline_model="anthropic/claude-haiku-4-5",
    # revision_types=["consistency", "conformance"],
    # max_revision_cycles=1,
    filtering_types=["conformance", "structural"],
    skip_outline=True
)
run_id = run.id
print(f"Run started — ID: {run_id}")


def run_not_ready(result):
    return result.status not in ("SUCCEEDED", "FAILED")


@retry(wait=wait_fixed(10), retry=retry_if_result(run_not_ready), stop=stop_never)
def poll_run(run_id):
    return df_client.dataframer.runs.retrieve(run_id=run_id)


run_result = poll_run(run_id)
if run_result.status == "FAILED":
    raise RuntimeError("Generation run failed")

print(f"\nRun complete")
print(f"Samples completed: {run_result.samples_completed}")
print(f"Samples failed:    {run_result.samples_failed}")

In [ ]:
def zip_not_ready(result):
    url = result if isinstance(result, str) else result.download_url
    return url is None


@retry(wait=wait_fixed(5), retry=retry_if_result(zip_not_ready), stop=stop_never)
def poll_download(run_id):
    return df_client.dataframer.runs.files.download_all(run_id=run_id)


dl = poll_download(run_id)
download_url = dl if isinstance(dl, str) else dl.download_url

output_dir = Path("output") / run_id
output_dir.mkdir(parents=True, exist_ok=True)

zip_data = requests.get(download_url).content
with zipfile.ZipFile(io.BytesIO(zip_data)) as zf:
    zf.extractall(output_dir)
    print(f"Extracted {len(zf.namelist())} files to {output_dir}/")

gen_dfs = [pd.read_csv(f) for f in sorted(output_dir.glob("*.csv"))]
gen_df = pd.concat(gen_dfs, ignore_index=True)
gen_df["Med_Keywords_Count"] = pd.to_numeric(gen_df["Med_Keywords_Count"], errors="coerce").fillna(0).astype(int)

print(f"\nGenerated population: {len(gen_df)} rows x {len(gen_df.columns)} columns")
gen_df.head(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df["Ins_Age"].dropna(), bins=30, alpha=0.6, color="#4C72B0", label="2015 (original)", density=True)
axes[0].hist(gen_df["Ins_Age"].dropna(), bins=30, alpha=0.6, color="#DD8452", label="2026 (generated)", density=True)
axes[0].set_title("Age Distribution")
axes[0].set_xlabel("Ins_Age (normalized)")
axes[0].set_ylabel("Density")
axes[0].legend()

axes[1].hist(df["BMI"].dropna(), bins=30, alpha=0.6, color="#4C72B0", label="2015 (original)", density=True)
axes[1].hist(gen_df["BMI"].dropna(), bins=30, alpha=0.6, color="#DD8452", label="2026 (generated)", density=True)
axes[1].set_title("BMI Distribution")
axes[1].set_xlabel("BMI (normalized)")
axes[1].set_ylabel("Density")
axes[1].legend()

max_kw = 14
axes[2].hist(
    df["Med_Keywords_Count"].clip(upper=max_kw), bins=range(max_kw + 2),
    alpha=0.6, color="#4C72B0", label="2015 (original)", density=True, align="left",
)
axes[2].hist(
    gen_df["Med_Keywords_Count"].clip(upper=max_kw), bins=range(max_kw + 2),
    alpha=0.6, color="#DD8452", label="2026 (generated)", density=True, align="left",
)
axes[2].set_title("Medical Keyword Count")
axes[2].set_xlabel("Med_Keywords_Count")
axes[2].set_ylabel("Density")
axes[2].legend()

plt.suptitle("Feature Distributions: 2015 (Original) vs 2026 (DataFramer-generated)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"Age    — 2015 mean: {df['Ins_Age'].mean():.3f}   2026 mean: {gen_df['Ins_Age'].mean():.3f}")
print(f"BMI    — 2015 mean: {df['BMI'].mean():.3f}   2026 mean: {gen_df['BMI'].mean():.3f}")
print(f"MedKw  — 2015 mean: {df['Med_Keywords_Count'].mean():.2f}   2026 mean: {gen_df['Med_Keywords_Count'].mean():.2f}")

## 6. Run Both Populations Through the 2015 Model

We apply the same frozen model — same weights, same 7 class thresholds calibrated on 2015 training data — to both populations and compare the risk class distributions.

In [ ]:
# Score 2015 population (full dataset)
X_2015 = engineer_features(df[FEATURE_COLS])
preds_2015 = model.predict(X_2015)
classes_2015 = predict_classes(preds_2015, THRESHOLDS)

# Score 2026 generated population
# reindex ensures we only pass the columns the model expects; missing ones become NaN -> -1
gen_features = gen_df.reindex(columns=FEATURE_COLS).copy()
X_2026 = engineer_features(gen_features)
preds_2026 = model.predict(X_2026)
classes_2026 = predict_classes(preds_2026, THRESHOLDS)

print(f"{'Class':<8} {'2015 %':>8} {'2026 %':>8} {'Change':>10}")
print("-" * 38)
for c in range(1, 9):
    p15 = (classes_2015 == c).mean() * 100
    p26 = (classes_2026 == c).mean() * 100
    delta = p26 - p15
    arrow = "up" if delta > 1 else ("down" if delta < -1 else "  ")
    print(f"  {c:<6} {p15:>7.1f}% {p26:>7.1f}%  {arrow} {abs(delta):>5.1f}pp")

high_2015 = (classes_2015 >= 7).mean() * 100
high_2026 = (classes_2026 >= 7).mean() * 100
print(f"\nHigh-risk (class 7-8): {high_2015:.1f}% -> {high_2026:.1f}%  ({high_2026 - high_2015:+.1f}pp)")
print(f"Mean raw score:        {preds_2015.mean():.3f} -> {preds_2026.mean():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

classes_range = list(range(1, 9))
pct_2015 = [(classes_2015 == c).mean() * 100 for c in classes_range]
pct_2026 = [(classes_2026 == c).mean() * 100 for c in classes_range]

x = np.arange(len(classes_range))
width = 0.38

axes[0].bar(x - width / 2, pct_2015, width, label="2015 (original)", color="#4C72B0", edgecolor="white")
axes[0].bar(x + width / 2, pct_2026, width, label="2026 (generated)", color="#DD8452", edgecolor="white")
axes[0].set_xticks(x)
axes[0].set_xticklabels([str(c) for c in classes_range])
axes[0].set_xlabel("Risk Class (1 = low, 8 = high)")
axes[0].set_ylabel("% of Applicants")
axes[0].set_title("Risk Class Distribution: 2015 vs 2026")
axes[0].legend()
axes[0].yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))

axes[1].hist(preds_2015, bins=60, alpha=0.6, color="#4C72B0", label="2015 (original)", density=True)
axes[1].hist(preds_2026, bins=60, alpha=0.6, color="#DD8452", label="2026 (generated)", density=True)
for t in THRESHOLDS:
    axes[1].axvline(t, color="gray", linestyle="--", linewidth=0.8, alpha=0.7)
axes[1].set_xlabel("Raw Model Score")
axes[1].set_ylabel("Density")
axes[1].set_title("Score Distribution (dashed = 2015 class thresholds)")
axes[1].legend()

plt.suptitle("2015 Model Applied to 2015 vs 2026 Applicant Populations", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 7. The Business Question

The 2015 model — weights and thresholds unchanged — classifies a larger share of the 2026 population into high-risk classes. The raw score distribution has shifted rightward against fixed thresholds.

There are two possible explanations, and the model itself cannot tell you which is true:

**Explanation A: the population is genuinely riskier.** Older applicants, higher BMI, more medical flags are real elevated-risk signals, and the model is correctly flagging them.

**Explanation B: the model is miscalibrated.** The feature-to-risk relationships have shifted since 2015. For example, a prescription that appeared in Medical_Keyword flags was rare and strongly correlated with mortality in 2015 — but in 2026 it is common and actively managed, meaning its risk signal has weakened. The model still applies the 2015 weight.

The carrier cannot distinguish these without re-validating the model against actual 2026 outcomes data. That investigation starts here — with the ability to construct this scenario in the first place.

**What DataFramer enables:**
- Build the shifted population without exposing real policyholder data
- Control exactly which distributions shift and by how much
- Reproduce the scenario reliably for multiple model versions or underwriting rules
- Do this in hours, not the months it would take to accumulate real data with the right distribution

The next step for the carrier: attach real outcomes to a validation cohort, compare predicted risk class to actual claim frequency by class, and determine whether Explanation A or B — or some combination — accounts for the shift.